# ML — Car Price Prediction

## Objectives
* Prepare data for machine learning by encoding categorical variables and scaling features
* Test multiple regression algorithms using quick search with default hyperparameters
* Identify the best performing algorithms for extensive hyperparameter optimisation
* Optimise the best model using GridSearchCV
* Evaluate model performance using R², RMSE and MAE on both train and test sets
* Identify the most important features for predicting car price
* Save the best model for use in the Streamlit app

## Inputs
* outputs/datasets/cleaned/car_prices_cleaned.csv


## Outputs
* Trained ML model saved to outputs/models/
* Model performance metrics (RMSE, R²)
* Feature importance visualisation

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2'

# Section 1 — Load Data

In [36]:
import pandas as pd
import numpy as np
import plotly.express as px
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from feature_engine.encoding import OrdinalEncoder

df = pd.read_csv('outputs/datasets/cleaned/car_prices_cleaned.csv')
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,CarBrand,price_per_horsepower,price_per_enginesize
0,1,3,alfa-romero giulia,gas,std,2,convertible,rwd,front,88.6,...,2.68,9.0,111,5000,21,27,13495.0,alfa-romero,121.576577,103.807692
1,2,3,alfa-romero stelvio,gas,std,2,convertible,rwd,front,88.6,...,2.68,9.0,111,5000,21,27,16500.0,alfa-romero,148.648649,126.923077
2,3,1,alfa-romero Quadrifoglio,gas,std,2,hatchback,rwd,front,94.5,...,3.47,9.0,154,5000,19,26,16500.0,alfa-romero,107.142857,108.552632
3,4,2,audi 100 ls,gas,std,4,sedan,fwd,front,99.8,...,3.40,10.0,102,5500,24,30,13950.0,audi,136.764706,127.981651
4,5,2,audi 100ls,gas,std,4,sedan,4wd,front,99.4,...,3.40,8.0,115,5500,18,22,17450.0,audi,151.739130,128.308824


---

# Section 2 — Data Preparation
We follow the CRISP-DM workflow and build a scikit-learn pipeline with:
* Feature Scaling — StandardScaler
* Feature Selection — SelectFromModel
* Model — tested with multiple algorithms

The target variable is **price** (continuous) — this is a **regression** task.

## CarBrand Extraction
CarBrand is extracted from CarName and cleaned before entering the pipeline.
The following typo corrections were applied to ensure brand names are 
consistent — without these corrections, misspelled names would be treated 
as separate categories by the model, leading to incorrect predictions.

* 'maxda' → 'mazda'
* 'porcshce' → 'porsche'
* 'toyouta' → 'toyota'
* 'vokswagen' / 'vw' → 'volkswagen'

In [5]:
# Extract CarBrand from CarName (moved here from ETL as per lecturer feedback)
df['CarBrand'] = df['CarName'].str.split().str[0].str.lower()
df['CarBrand'] = df['CarBrand'].replace({
    'maxda': 'mazda',
    'porcshce': 'porsche',
    'toyouta': 'toyota',
    'vokswagen': 'volkswagen',
    'vw': 'volkswagen'
})

In [6]:
# Drop columns not useful for ML
features = ['enginesize', 'horsepower', 'curbweight', 'carwidth', 
            'carlength', 'wheelbase', 'boreratio', 'stroke',
            'compressionratio', 'peakrpm', 'citympg', 'highwaympg',
            'fueltype', 'aspiration', 'doornumber', 'carbody', 
            'drivewheel', 'enginelocation', 'enginetype', 
            'cylindernumber', 'fuelsystem', 'CarBrand']

X = df[features]
y = df['price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget variable: price")
print(f"Mean price: ${y.mean():.2f}")

Features shape: (205, 22)
Target shape: (205,)

Target variable: price
Mean price: $13276.71


In [7]:
categorical_cols = ['fueltype', 'aspiration', 'doornumber', 'carbody', 
                    'drivewheel', 'enginelocation', 'enginetype', 
                    'cylindernumber', 'fuelsystem', 'CarBrand']

numerical_cols = ['enginesize', 'horsepower', 'curbweight', 'carwidth', 
                  'carlength', 'wheelbase', 'boreratio', 'stroke',
                  'compressionratio', 'peakrpm', 'citympg', 'highwaympg']

In [8]:
# Ensure categorical columns are of type object
X = df[features].copy()
for col in categorical_cols:
    X[col] = X[col].astype(str)

## Train/Test Split
80% of the data is used to train the model, 20% is held back to test 
how the model performs on unseen data. random_state=0 ensures 
reproducibility — the same split every time.

In [37]:
encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=categorical_cols)

X_encoded = encoder.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=0)

print(f"* Train set: {X_train.shape} {y_train.shape}")
print(f"* Test set: {X_test.shape} {y_test.shape}")

* Train set: (164, 22) (164,)
* Test set: (41, 22) (41,)


/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/.venv/lib/python3.11/site-packages/feature_engine/encoding/base_encoder.py:223: FutureWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead

/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/.venv/lib/python3.11/site-packages/feature_engine/encoding/base_encoder.py:223: FutureWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead

/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/.venv/lib/python3.11/site-packages/feature_engine/encoding/base_encoder.py:223: FutureWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead

/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_2/.venv/lib/python3.11/site-packages/feature_engine/encoding/base_encoder.py:223: FutureWarning:

is_ca

---

# Section 3 — ML Pipeline

Following the scikit-learn pipeline workflow, we define 
a pipeline with three steps:
1. Feature encoding — OrdinalEncoder for categorical variables
2. Feature scaling — StandardScaler for numerical variables  
3. Feature selection — SelectFromModel
4. Model

In [38]:
def PipelineRegression(model):
    pipeline_base = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('feat_scaling', StandardScaler()),
        ('feat_selection', SelectFromModel(model)),
        ('model', model),
    ])
    return pipeline_base

---

# Section 4 — Hyperparameter Optimisation

We use the HyperparameterOptimizationSearch class to 
test multiple regression algorithms simultaneously.

Strategy (two-step search):
1. Quick search with default hyperparameters on all algorithms
2. Extensive search on the best performing algorithms

In [39]:
class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")
            model = PipelineRegression(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring)
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                 'estimator': key,
                 'min_score': min(scores),
                 'max_score': max(scores),
                 'mean_score': np.mean(scores),
                 'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))
            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

---

## Define Algorithms and Hyperparameters

## Step 1 — Run Quick Search with Default Hyperparameters
All algorithms are tested with default hyperparameters to identify 
which ones best fit the car price data.

In [40]:
models_search = {
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

params_search = {
    "LinearRegression": {},
    "DecisionTreeRegressor": {},
    "RandomForestRegressor": {},
    "GradientBoostingRegressor": {},
    "ExtraTreesRegressor": {},
}

In [25]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring='r2',
           n_jobs=-2,
           cv=5)


Running GridSearchCV for LinearRegression 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for DecisionTreeRegressor 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for RandomForestRegressor 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for GradientBoostingRegressor 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for ExtraTreesRegressor 

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [26]:
score_summary, _ = search.score_summary(sort_by='mean_score')
score_summary[['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']]

,estimator,min_score,mean_score,max_score,std_score
4,ExtraTreesRegressor,0.870246,0.908351,0.951437,0.032546
3,GradientBoostingRegressor,0.844708,0.908135,0.936311,0.033485
2,RandomForestRegressor,0.843255,0.888813,0.917493,0.02542
0,LinearRegression,0.799109,0.847415,0.899364,0.039293
1,DecisionTreeRegressor,0.670327,0.806097,0.853641,0.06902


### Quick Search Results
ExtraTreesRegressor and GradientBoostingRegressor perform best with 
a mean R2 score of approximately 0.91, meaning the models explain 91% of the 
variance in car prices.

These two algorithms will be taken forward for extensive 
hyperparameter optimisation in Step 2.

---

## Step 2 — Extensive Hyperparameter Search
Based on the quick search results, we run an extensive hyperparameter 
search on the two best performing algorithms: ExtraTreesRegressor 
and GradientBoostingRegressor.

In [27]:
models_search_2 = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
}

params_search_2 = {
    "ExtraTreesRegressor": {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 4],
    },
    "GradientBoostingRegressor": {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1, 0.2],
        'model__max_depth': [3, 5],
    }
}

search_2 = HyperparameterOptimizationSearch(
    models=models_search_2, 
    params=params_search_2)

search_2.fit(X_train, y_train,
             scoring='r2',
             n_jobs=-2,
             cv=5)


Running GridSearchCV for ExtraTreesRegressor 

Fitting 5 folds for each of 27 candidates, totalling 135 fits

Running GridSearchCV for GradientBoostingRegressor 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [28]:
score_summary_2, _ = search_2.score_summary(sort_by='mean_score')
score_summary_2[['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']]

,estimator,min_score,mean_score,max_score,std_score
32,GradientBoostingRegressor,0.846294,0.910132,0.940536,0.03379
9,ExtraTreesRegressor,0.877066,0.909323,0.950279,0.029241
19,ExtraTreesRegressor,0.864467,0.909218,0.955756,0.034695
1,ExtraTreesRegressor,0.864467,0.909218,0.955756,0.034695
11,ExtraTreesRegressor,0.866926,0.908605,0.953324,0.032977
10,ExtraTreesRegressor,0.870407,0.908455,0.951092,0.031612
18,ExtraTreesRegressor,0.870246,0.908351,0.951437,0.032546
0,ExtraTreesRegressor,0.870246,0.908351,0.951437,0.032546
2,ExtraTreesRegressor,0.86114,0.908228,0.95574,0.035895
20,ExtraTreesRegressor,0.86114,0.908228,0.95574,0.035895


In [29]:
best_model_key = score_summary_2.iloc[0]['estimator']
best_params = search_2.grid_searches[best_model_key].best_params_
best_estimator = search_2.grid_searches[best_model_key].best_estimator_

print(f"Best model: {best_model_key}")
print(f"Best parameters: {best_params}")

Best model: GradientBoostingRegressor
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}


### Best Model
The best performing model is GradientBoostingRegressor with the following 
hyperparameters selected automatically by GridSearchCV:
* learning_rate: 0.1
* max_depth: 3
* n_estimators: 200

In [41]:
# from LMS SciKit-Learn Topic 7 Cross Validation Search

def regression_performance(X_train, y_train, X_test, y_test, pipeline):
    print("Model Evaluation \n")
    print("* Train Set")
    regression_evaluation(X_train, y_train, pipeline)
    print("* Test Set")
    regression_evaluation(X_test, y_test, pipeline)

def regression_evaluation(X, y, pipeline):
    prediction = pipeline.predict(X)
    print(f"  R2 Score: {r2_score(y, prediction).round(3)} \n")
    print(f"  Mean Absolute Error: {mean_absolute_error(y, prediction).round(3)} \n")
    print(f"  Mean Squared Error: {mean_squared_error(y, prediction).round(3)} \n")
    print(f"  Root Mean Squared Error: {np.sqrt(mean_squared_error(y, prediction)).round(3)} \n")

In [33]:
regression_performance(X_train, y_train, X_test, y_test, best_estimator)

Model Evaluation 

* Train Set
  R2 Score: 0.996 

  Mean Absolute Error: 351.003 

  Mean Squared Error: 232272.219 

  Root Mean Squared Error: 481.946 

* Test Set
  R2 Score: 0.916 

  Mean Absolute Error: 1844.508 

  Mean Squared Error: 6490162.051 

  Root Mean Squared Error: 2547.58 



### Model Performance

**Train Set:**
* R2: 0.996 — the model explains 99.6% of variance in training data
* RMSE: $482 — average prediction error on training data

**Test Set:**
* R2: 0.916 — the model explains 91.6% of variance in unseen data
* RMSE: $2,548 — average prediction error on unseen data

**Note — Overfitting:**
The large gap between train (0.996) and test (0.916) R2 scores suggests 
the model is slightly overfitting — it performs better on data it has 
seen during training than on unseen data. This could be addressed with further regularisation.

**Note on overfitting:**
The gap between train R2 (0.996) and test R2 (0.916) indicates mild overfitting. 
This is partly expected given the small dataset size (205 rows). 
The test R2 of 0.916 is still considered a strong result.

---

## Feature Importance
Feature importance shows which variables have the greatest influence 
on the model's price predictions. This helps us understand which car 
attributes matter most when determining price.

In [34]:
feature_importance = best_estimator.named_steps['model'].feature_importances_
feature_names = best_estimator.named_steps['feat_selection'].get_support()

# Get selected feature names
selected_features = X_train.columns[feature_names]

importance_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

fig = px.bar(importance_df, x='Importance', y='Feature',
             orientation='h',
             title='Feature Importance — GradientBoostingRegressor',
             labels={'Importance': 'Importance Score', 'Feature': 'Feature'})
fig.show()

### Feature Importance Results

The model selected 4 features as most important for predicting car price:

1. enginesize (0.60) — by far the strongest predictor, confirming Hypothesis 1
2. curbweight (0.30) — second strongest, consistent with our EDA finding that 
   curbweight unexpectedly outperformed horsepower
3. horsepower (0.13) — third, also confirmed in Hypothesis 1
4. cylindernumber (0.05) — smallest contribution but still selected by the model

**Key insight:** The ML model confirms our EDA findings — enginesize is the 
strongest predictor of car price, followed by curbweight and horsepower.
Only 4 out of 22 features were selected by SelectFromModel, suggesting 
the other features add little predictive value.

---

## Save Model
The best model is saved to outputs/models/ so it can be loaded 
and used in the Streamlit app for price predictions.

In [35]:
os.makedirs('outputs/models', exist_ok=True)
joblib.dump(best_estimator, 'outputs/models/car_price_model.pkl')
print("Model saved successfully!")

Model saved successfully!
